[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pflahert/hd-stats-workshop/blob/main/notebooks/lab3.ipynb)

# Lab 3: The Map That Drew Itself
## Dimension Reduction

**Day 2, 9:00–10:00** | Learning Outcomes:
- **LO2** — apply dimension reduction (PCA / UMAP / t-SNE) to high-dimensional biological data.
- **LO3** — critically evaluate analysis code and AI-generated output.

This lab applies **Lecture 3** (estimating low-dimensional structure) to the Golub leukemia dataset. Recall the Day 1 arc: Lecture 1 estimated which genes to call significant (testing); Lecture 2 estimated quantities from the histogram (estimation). Day 2 continues the estimation theme on different objects — today's object is the *low-dimensional manifold* on which the data approximately live.

In this lab you will:
1. Perform PCA on the Golub leukemia dataset and interpret the results
2. Evaluate preprocessing choices (centering, scaling)
3. Apply t-SNE and UMAP — and see them lie
4. Test parameter sensitivity
5. Compare methods and articulate what each can and cannot tell you

::: {.callout-note}
## How this lab uses AI

This lab integrates AI at several steps, not just at the end. Two kinds of AI exercises appear:

- **Single-shot critiques** — you prompt, paste the response, and evaluate it against named criteria.
- **Multi-turn conversations** — you prompt, critique the response, write a follow-up prompt that addresses your critique, paste the refined response, and write a final assessment. *At least one multi-turn exercise per lab is required.* In this lab, the **Part 5 (UMAP on noise)** exercise is the multi-turn one.

Use any AI assistant (the [UMass campus GenAI platform](https://www.umass.edu/it/genai/platform), Claude, ChatGPT, or another). Note which AI you used for each prompt.
:::

The key question: **can we see the ALL vs. AML distinction in lower dimensions?** A subsidiary question — also answered here — is *whether the apparent separation in the low-dimensional plot is real or an artifact of the embedding procedure*.

In [ ]:
# Setup
!pip install -q umap-learn

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
from umap import UMAP   # umap-learn package; using `from umap import UMAP` fails fast if the wrong PyPI package is installed.

# Workshop convention: deterministic seed.
np.random.seed(2026)

try:
    plt.style.use('seaborn-v0_8-whitegrid')
except OSError:
    plt.style.use('default')

plt.rcParams['figure.figsize'] = (8, 6)
plt.rcParams['font.size'] = 12


In [ ]:
# Load the Golub expression matrix and metadata from the workshop repo (CSV — same source as Labs 1, 2).
expr_url = "https://raw.githubusercontent.com/pflahert/hd-stats-workshop/v2026-spring-data/data/golub_expression.csv"
meta_url = "https://raw.githubusercontent.com/pflahert/hd-stats-workshop/v2026-spring-data/data/golub_metadata.csv"

expr_genes_x_samples = pd.read_csv(expr_url, index_col=0)
meta = pd.read_csv(meta_url)

assert expr_genes_x_samples.shape == (3051, 72), \
    f"Unexpected expression shape: {expr_genes_x_samples.shape} (expected (3051, 72))"
assert list(expr_genes_x_samples.columns) == list(meta["sample_id"]), \
    "Sample IDs do not match between expression and metadata."

# Samples x genes for sklearn PCA / UMAP.
X = expr_genes_x_samples.values.T
gene_names = expr_genes_x_samples.index.values
bt_type = meta["subtype"].values
color_map = {'ALL': '#2166ac', 'AML': '#b2182b'}   # blue=ALL, red=AML
colors = [color_map[t] for t in bt_type]

print(f"Data dimensions (samples x genes): {X.shape}")
print(f"Groups: ALL={int((bt_type=='ALL').sum())}, AML={int((bt_type=='AML').sum())}")


---

## Part 1: PCA

Perform PCA on the Golub expression matrix. The data should be **centered and scaled** (each gene standardized to mean 0, variance 1). This ensures that genes with larger raw variance do not dominate the first principal components.

We will produce:
1. A **scree plot** showing proportion of variance explained for the first 30 PCs
2. A **cumulative variance** plot — marking where 50% and 80% are reached
3. A **scatter plot** of PC1 vs. PC2, colored by ALL/AML subtype

In [ ]:
np.random.seed(2026)

# Standardize the data: center and scale each gene
scaler = StandardScaler()  # mean=0, std=1 per feature
X_scaled = scaler.fit_transform(X)

# Fit PCA (keep all components up to min(n, p))
pca = PCA(random_state=2026)
pca_scores = pca.fit_transform(X_scaled)

# Variance explained
var_explained = pca.explained_variance_ratio_
cum_var = np.cumsum(var_explained)

# --- Three-panel figure ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Scree plot (first 30 PCs)
ax = axes[0]
ax.bar(range(1, 31), var_explained[:30] * 100, color='steelblue', edgecolor='white')
ax.set_xlabel('Principal Component')
ax.set_ylabel('Variance Explained (%)')
ax.set_title('Scree Plot')
ax.set_xlim(0.5, 30.5)

# 2. Cumulative variance
ax = axes[1]
ax.plot(range(1, len(cum_var) + 1), cum_var * 100, 'o-', markersize=2, color='steelblue')
# Mark 50% and 80% thresholds
n_50 = np.argmax(cum_var >= 0.50) + 1
n_80 = np.argmax(cum_var >= 0.80) + 1
ax.axhline(50, color='orange', linestyle='--', linewidth=1, label=f'50% at PC{n_50}')
ax.axhline(80, color='red', linestyle='--', linewidth=1, label=f'80% at PC{n_80}')
ax.axvline(n_50, color='orange', linestyle=':', linewidth=1, alpha=0.5)
ax.axvline(n_80, color='red', linestyle=':', linewidth=1, alpha=0.5)
ax.set_xlabel('Number of PCs')
ax.set_ylabel('Cumulative Variance Explained (%)')
ax.set_title('Cumulative Variance')
ax.set_xlim(0, 72)
ax.legend(fontsize=10)

# 3. PC1 vs PC2 scatter
ax = axes[2]
for label in ['ALL', 'AML']:
    mask = bt_type == label
    ax.scatter(pca_scores[mask, 0], pca_scores[mask, 1],
               c=color_map[label], label=label, alpha=0.7, edgecolors='white', s=50)
ax.set_xlabel(f'PC1 ({var_explained[0]*100:.1f}%)')
ax.set_ylabel(f'PC2 ({var_explained[1]*100:.1f}%)')
ax.set_title('PC1 vs PC2')
ax.legend(title='Subtype')

plt.tight_layout()
plt.show()

# Report
print(f"PC1 explains {var_explained[0]*100:.2f}% of variance")
print(f"PC2 explains {var_explained[1]*100:.2f}% of variance")
print(f"PCs needed for 50% variance: {n_50}")
print(f"PCs needed for 80% variance: {n_80}")

### Critique Checklist

| Question | Your Answer |
|----------|-------------|
| Did you use `center=True` and `scale=True` (StandardScaler)? | |
| Does PC1 separate ALL from AML? What % variance does it explain? | |
| Is PC1 substantially larger than $1/(n-1) \approx 1.41\%$ (the naive per-PC share when all $n-1$ nonzero eigenvalues are equal)? | |

### Exercise 1.2

With $p = 3{,}051$ and $n = 72$, only $\min(n-1, p) = 71$ eigenvalues of the sample covariance are nonzero, so the "naive per-PC share" under uniform variance is $1/(n-1) \approx 1.41\%$. Is PC1 substantially larger than this? *(Caveat: under the Marchenko–Pastur law, even pure-noise eigenvalues at this $\gamma = p/n \approx 42$ are spread up to $(1+\sqrt{\gamma})^2 \approx 57$ on the standardized-data scale, so "substantially larger than 1.41%" is not by itself proof of signal — the permutation test in Part 3 is the diagnostic.)*

**YOUR ANSWER:**


### Exercise 1.3 — AI on standardization (single-shot)

Ask an AI assistant whether to standardize the data before PCA on the Golub dataset, and check whether it names the actual failure mode.

> "Reply in three to four sentences only (no .docx file). I want to apply PCA to a microarray gene-expression matrix (3,051 genes × 72 samples) and identify the directions of greatest variation. Should I standardize each gene (center to mean 0, scale to unit variance) before running PCA, or use the raw expression values? What happens if I don't standardize? Name the failure mode explicitly and cite the relevant `sklearn` preprocessing class."

**AI's response (paste here):**


**Your critique** — common failure modes:

| Criterion | Pass / Fail / Partial | Notes |
|---|---|---|
| Did the AI name the failure mode (genes with larger raw variance dominate the leading PCs)? | | |
| Did the AI cite `sklearn.preprocessing.StandardScaler` (or equivalent)? | | |
| Did the AI distinguish centering (mean 0) from standardization (mean 0 + var 1)? | | |
| Did the AI avoid claiming standardization is "optional" without qualification? | | |

**One-sentence overall assessment** — did the AI identify the actual failure mode of skipping standardization on microarray data?

---

## Part 2: Loadings

Extract the loadings for PC1. Which genes drive the separation between ALL and AML?

In [ ]:
# Extract PC1 loadings
pc1_loadings = pca.components_[0]  # shape: (p,)

# Create a Series for easy sorting
loadings_series = pd.Series(pc1_loadings, index=gene_names)

# Top 10 positive and top 10 negative loadings
top_pos = loadings_series.nlargest(10)
top_neg = loadings_series.nsmallest(10)

print("=== Top 10 Positive PC1 Loadings (drive ALL direction) ===")
print(pd.DataFrame({'Gene': top_pos.index, 'Loading': top_pos.values}).to_string(index=False))
print()
print("=== Top 10 Negative PC1 Loadings (drive AML direction) ===")
print(pd.DataFrame({'Gene': top_neg.index, 'Loading': top_neg.values}).to_string(index=False))

# Histogram of all PC1 loadings
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(pc1_loadings, bins=80, color='steelblue', edgecolor='white', alpha=0.8)
ax.axvline(0, color='black', linewidth=1)
ax.set_xlabel('PC1 Loading')
ax.set_ylabel('Number of Genes')
ax.set_title('Distribution of PC1 Loadings (3,051 genes)')
plt.tight_layout()
plt.show()

# Note
print(f"\nMost loadings are near zero. Only a subset of genes drive the PC1 separation.")
print(f"Loadings > |0.03|: {np.sum(np.abs(pc1_loadings) > 0.03)} genes")

### Exercise 2.1

Are most loadings near zero? Is the signal in PC1 driven by a few genes or distributed across many?

**YOUR ANSWER:**

---

## Part 3: How Many Components?

How do you decide how many PCs to keep? The scree plot gives a visual heuristic, but we can do better. A **permutation test** compares observed eigenvalues against eigenvalues from shuffled (null) data. PCs whose eigenvalues exceed the null distribution are "significant."

We also compute the **Marchenko-Pastur upper edge**: the largest eigenvalue expected from a pure-noise random matrix.

In [ ]:
np.random.seed(2026)

n_perms = 500          # 500 perms gives a stable 95th percentile
n_pcs_check = 20
n_samples, n_genes = X_scaled.shape

# Observed eigenvalues
observed_eigenvalues = pca.explained_variance_[:n_pcs_check]

# Permutation: shuffle each column independently to destroy cross-gene structure, refit PCA
perm_eigenvalues = np.zeros((n_perms, n_pcs_check))
for i in range(n_perms):
    X_perm = X_scaled.copy()
    for j in range(X_perm.shape[1]):
        np.random.shuffle(X_perm[:, j])
    pca_perm = PCA(n_components=n_pcs_check, random_state=2026)
    pca_perm.fit(X_perm)
    perm_eigenvalues[i] = pca_perm.explained_variance_

perm_95 = np.percentile(perm_eigenvalues, 95, axis=0)
n_sig_pcs = int(np.sum(observed_eigenvalues > perm_95))

# Marchenko–Pastur upper edge for the sample covariance with gamma = p/n.
# Under standardization (unit-variance columns), MP predicts noise eigenvalues
# in [(1 - sqrt(gamma))^2, (1 + sqrt(gamma))^2]. Eigenvalues exceeding the upper
# edge are candidates for real signal. The BBP detection threshold for a
# population spike is ell > sqrt(gamma); a spike that strong produces a sample
# eigenvalue (1+ell)(1+gamma/ell) > (1+sqrt(gamma))^2.
gamma = n_genes / n_samples
mp_upper = (1 + np.sqrt(gamma))**2
bbp_spike = np.sqrt(gamma)

# Plot
fig, ax = plt.subplots(figsize=(10, 6))
pcs = np.arange(1, n_pcs_check + 1)
ax.plot(pcs, observed_eigenvalues, 'o-', color='steelblue', linewidth=2,
        markersize=6, label='Observed eigenvalues')
ax.plot(pcs, perm_95, 's--', color='#b2182b', linewidth=2,
        markersize=6, label=f'95th percentile (permutation, n={n_perms})')
ax.axhline(mp_upper, color='orange', linestyle=':', linewidth=2,
           label=f'MP upper edge $(1+\sqrt{{\gamma}})^2 \approx$ {mp_upper:.1f}')
ax.set_xlabel('Principal Component')
ax.set_ylabel('Eigenvalue')
ax.set_title(f'Observed vs. Null Eigenvalues ($\gamma = p/n \approx$ {gamma:.1f})')
ax.legend(fontsize=10)
ax.set_xticks(pcs)
plt.tight_layout()
plt.show()

print(f"Number of significant PCs (above permutation 95th percentile): {n_sig_pcs}")
print(f"Marchenko-Pastur upper edge:        {mp_upper:.1f}")
print(f"BBP detection threshold on spike:   ell > sqrt(gamma) = {bbp_spike:.2f}")
print(f"Observed leading eigenvalue:        {observed_eigenvalues[0]:.1f}")
print(f"  ({'above' if observed_eigenvalues[0] > mp_upper else 'within'} MP bulk; "
      f"{'detectable' if observed_eigenvalues[0] > mp_upper else 'within'} per BBP)")


---

## Part 4: UMAP

Apply UMAP to the Golub data. **Critical**: run UMAP on the top 30 PCs, not the raw 3,051 dimensions. Two reasons:
1. **Computational**: UMAP on 3,051 features is slower and noisier
2. **Statistical**: most of those 3,051 dimensions are noise — PCA denoises first

In [ ]:
np.random.seed(2026)

# Get top 30 PCs from the standardized data
pca_30 = PCA(n_components=30, random_state=2026)
X_pca30 = pca_30.fit_transform(X_scaled)
print(f"Input to UMAP: {X_pca30.shape[0]} samples x {X_pca30.shape[1]} PCs")

# Run UMAP on the PC scores
reducer = UMAP(random_state=2026)
X_umap = reducer.fit_transform(X_pca30)

# Plot UMAP embedding
fig, ax = plt.subplots(figsize=(8, 6))
for label in ['ALL', 'AML']:
    mask = bt_type == label
    ax.scatter(X_umap[mask, 0], X_umap[mask, 1],
               c=color_map[label], label=label, alpha=0.7,
               edgecolors='white', s=60)
ax.set_xlabel('UMAP 1')
ax.set_ylabel('UMAP 2')
ax.set_title('UMAP on Top 30 PCs')
ax.legend(title='Subtype')
plt.tight_layout()
plt.show()

### Exercise 4.1

Did you run UMAP on PCs or raw data? Why does this matter? (Two reasons: computational and statistical)

**YOUR ANSWER:**

---

## Part 5: UMAP on Noise

What happens when you run UMAP on pure noise? This is the critical lesson. UMAP (and t-SNE) can create apparent structure from nothing.

In [ ]:
# Use a local rng so this cell is reproducible even if students re-run it
# out of order with the global RNG in a different state.
local_rng = np.random.default_rng(2026)

# Three UMAP embeddings of the SAME pure-noise input at three different seeds.
# The lesson is not "clusters from nothing" (at n = 72 the noise embedding is a
# diffuse blob at default parameters) but "instability" — the same noise input
# deforms unpredictably across seeds, while a real signal is robust.
X_noise = local_rng.standard_normal((72, 30))
noise_labels = local_rng.choice(['ALL', 'AML'], size=72, p=[0.5, 0.5])

fig, axes = plt.subplots(1, 4, figsize=(20, 5))

# Panel 1: real data, real labels (reference)
ax = axes[0]
for label in ['ALL', 'AML']:
    mask = bt_type == label
    ax.scatter(X_umap[mask, 0], X_umap[mask, 1],
               c=color_map[label], label=label, alpha=0.7,
               edgecolors='white', s=50)
ax.set_xlabel('UMAP 1'); ax.set_ylabel('UMAP 2')
ax.set_title('REAL data, real labels')
ax.legend(title='Subtype')

# Panels 2-4: same noise input, three different UMAP seeds
for k, seed in enumerate([1, 2, 3]):
    ax = axes[k + 1]
    reducer_noise = UMAP(n_neighbors=15, min_dist=0.3, random_state=seed)
    X_umap_noise = reducer_noise.fit_transform(X_noise)
    for label in ['ALL', 'AML']:
        mask = noise_labels == label
        ax.scatter(X_umap_noise[mask, 0], X_umap_noise[mask, 1],
                   c=color_map[label], label=label, alpha=0.7,
                   edgecolors='white', s=50)
    ax.set_xlabel('UMAP 1'); ax.set_ylabel('UMAP 2')
    ax.set_title(f'NOISE input, UMAP seed={seed}')
    ax.legend(title='Random labels', fontsize=8)

plt.suptitle('UMAP on noise: the same pure-noise input deforms unpredictably across seeds.\n'
             'Real signal (left) is robust; artifactual structure (right three) is not.',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("Key insight: the rightmost three panels share an identical input matrix")
print("of pure standard-normal noise. They differ only in the UMAP random seed.")
print("Real biological signal would survive seed variation; pure-noise structure does not.")


### Exercise 5.1 — Multi-turn: why does UMAP look unstable on pure noise? *(required multi-turn)*

This is the multi-turn AI exercise for Lab 3. The three-panel UMAP-on-noise plot above shows that the *same* pure-noise input produces *different* 2D embeddings at different random seeds, while the real-data embedding is robust. Why? **Three turns of model output in the same chat session** so the AI can build on (and revise) its earlier answer.

**Turn 1 — initial prompt.** Ask the AI:

> "Reply in three short paragraphs only (no .docx file). I ran UMAP three times on the *same* $72 \times 30$ matrix of independent standard-normal noise, at three different random seeds (n_neighbors=15, min_dist=0.3). The three 2D embeddings look qualitatively different, while UMAP on a real signal-bearing matrix at the same parameters and three seeds looks essentially the same up to rotation/reflection. Why is UMAP's output unstable across seeds on pure noise but stable on real signal?"

**AI's response (Turn 1, paste here):**


**Your critique (write your own — do not copy the bullets verbatim).** Common failure modes to look for:

- The AI says only "UMAP is non-deterministic" without explaining *why* noise vs. signal differ in seed-sensitivity.
- The AI confuses UMAP's stochasticity (random initialization, edge sampling) with its tendency to produce *consistent* layouts on real data.
- The AI does not mention the *local k-nearest-neighbor graph* that UMAP builds from the high-D affinities, and how that graph is well-determined when there is true low-dimensional structure but ambiguous when there isn't.
- The AI does not propose an empirical test the lab already runs (compare embeddings across seeds).

**YOUR CRITIQUE OF THE TURN-1 RESPONSE:**


**Turn 2 — your follow-up prompt.** Based on the specific weaknesses you flagged, write a prompt in your own words asking the AI to give the technical mechanism. Don't copy a canned phrase — write what *you* want explained.

**YOUR TURN-2 PROMPT:**

**AI's response (Turn 2):**


**Turn 3 — actionable diagnostic.** In the same chat session, ask the AI for the specific empirical diagnostic to apply *before reporting any UMAP cluster claim*. The right answer is "run the same embedding at 3+ different seeds and check that the qualitative structure persists" — but make the AI say it.

**YOUR TURN-3 PROMPT:**

**AI's response (Turn 3):**


**Final assessment.** Did the Turn-3 response give you a concrete diagnostic protocol (run multiple seeds; report only structure that persists)? If still vague, what's the right one-sentence protocol in your own words?

---

## Part 6: Parameter Sensitivity

How much do UMAP and t-SNE results depend on their tuning parameters? We test a range of `n_neighbors` (UMAP) and `perplexity` (t-SNE).

In [ ]:
# UMAP: vary n_neighbors
n_neighbors_vals = [5, 15, 30, 50]

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.ravel()

for i, nn in enumerate(n_neighbors_vals):
    reducer_nn = UMAP(n_neighbors=nn, random_state=2026)
    X_umap_nn = reducer_nn.fit_transform(X_pca30)
    ax = axes[i]
    for label in ['ALL', 'AML']:
        mask = bt_type == label
        ax.scatter(X_umap_nn[mask, 0], X_umap_nn[mask, 1],
                   c=color_map[label], label=label, alpha=0.7,
                   edgecolors='white', s=40)
    ax.set_title(f'UMAP: n_neighbors = {nn}', fontsize=12)
    ax.set_xlabel('UMAP 1')
    ax.set_ylabel('UMAP 2')
    ax.legend(title='Subtype', fontsize=9)

plt.suptitle('UMAP Parameter Sensitivity (n_neighbors)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# t-SNE: vary perplexity
perplexity_vals = [5, 15, 30, 50]

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.ravel()

for i, perp in enumerate(perplexity_vals):
    tsne = TSNE(n_components=2, perplexity=perp, random_state=2026)
    X_tsne_perp = tsne.fit_transform(X_pca30)
    ax = axes[i]
    for label in ['ALL', 'AML']:
        mask = bt_type == label
        ax.scatter(X_tsne_perp[mask, 0], X_tsne_perp[mask, 1],
                   c=color_map[label], label=label, alpha=0.7,
                   edgecolors='white', s=40)
    ax.set_title(f't-SNE: perplexity = {perp}', fontsize=12)
    ax.set_xlabel('t-SNE 1')
    ax.set_ylabel('t-SNE 2')
    ax.legend(title='Subtype', fontsize=9)

plt.suptitle('t-SNE Parameter Sensitivity (perplexity)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### Exercise 6.1

How does perplexity affect the t-SNE embedding? Which perplexity gives the clearest separation? How stable are the results across different seeds?

**YOUR ANSWER:**

In [ ]:
# Reproducibility test: t-SNE with different seeds vs. PCA (deterministic)
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# t-SNE with seed 42
tsne_1 = TSNE(n_components=2, perplexity=30, random_state=42)
X_tsne_1 = tsne_1.fit_transform(X_pca30)
ax = axes[0, 0]
for label in ['ALL', 'AML']:
    mask = bt_type == label
    ax.scatter(X_tsne_1[mask, 0], X_tsne_1[mask, 1],
               c=color_map[label], label=label, alpha=0.7, edgecolors='white', s=40)
ax.set_title('t-SNE (seed=42)', fontsize=12)
ax.set_xlabel('t-SNE 1')
ax.set_ylabel('t-SNE 2')
ax.legend(title='Subtype', fontsize=9)

# t-SNE with seed 99
tsne_2 = TSNE(n_components=2, perplexity=30, random_state=99)
X_tsne_2 = tsne_2.fit_transform(X_pca30)
ax = axes[0, 1]
for label in ['ALL', 'AML']:
    mask = bt_type == label
    ax.scatter(X_tsne_2[mask, 0], X_tsne_2[mask, 1],
               c=color_map[label], label=label, alpha=0.7, edgecolors='white', s=40)
ax.set_title('t-SNE (seed=99)', fontsize=12)
ax.set_xlabel('t-SNE 1')
ax.set_ylabel('t-SNE 2')
ax.legend(title='Subtype', fontsize=9)

# PCA run 1
ax = axes[1, 0]
for label in ['ALL', 'AML']:
    mask = bt_type == label
    ax.scatter(pca_scores[mask, 0], pca_scores[mask, 1],
               c=color_map[label], label=label, alpha=0.7, edgecolors='white', s=40)
ax.set_title('PCA (run 1 -- deterministic)', fontsize=12)
ax.set_xlabel(f'PC1 ({var_explained[0]*100:.1f}%)')
ax.set_ylabel(f'PC2 ({var_explained[1]*100:.1f}%)')
ax.legend(title='Subtype', fontsize=9)

# PCA run 2 (identical — PCA is deterministic given the same input)
pca_2 = PCA(random_state=2026)
pca_scores_2 = pca_2.fit_transform(X_scaled)
ax = axes[1, 1]
for label in ['ALL', 'AML']:
    mask = bt_type == label
    ax.scatter(pca_scores_2[mask, 0], pca_scores_2[mask, 1],
               c=color_map[label], label=label, alpha=0.7, edgecolors='white', s=40)
ax.set_title('PCA (run 2 -- deterministic)', fontsize=12)
ax.set_xlabel(f'PC1 ({var_explained[0]*100:.1f}%)')
ax.set_ylabel(f'PC2 ({var_explained[1]*100:.1f}%)')
ax.legend(title='Subtype', fontsize=9)

plt.suptitle('Reproducibility: t-SNE (stochastic) vs. PCA (deterministic)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("t-SNE gives DIFFERENT plots with different seeds.")
print("PCA gives the SAME plot every time.")

---

## Part 7: AI Extension

Ask an AI to run t-SNE and UMAP on the Golub leukemia dataset and produce a side-by-side comparison plot colored by ALL vs. AML subtype. **The point of this exercise is to test whether the AI picks the right preprocessing — PCA-first reduction to ~30 dimensions before any nonlinear embedding — without being told to.** The prompt below deliberately *does not* mention PCA.

**Suggested prompt** (paste-ready, with format directive):

> "Reply with a single Python code block (no prose, no .docx file, no surrounding explanation — just runnable code I can paste into a Colab cell).
>
> The Golub 1999 leukemia microarray dataset has 3,051 genes × 72 samples with class labels ALL or AML. Load it from `https://raw.githubusercontent.com/pflahert/hd-stats-workshop/v2026-spring-data/data/golub_expression.csv` (genes × samples — you will need to transpose) and `https://raw.githubusercontent.com/pflahert/hd-stats-workshop/v2026-spring-data/data/golub_metadata.csv` (sample_id, subtype). Standardize the gene matrix appropriately, then produce a side-by-side comparison plot of t-SNE and UMAP embeddings of the data, colored by ALL vs. AML. Use `sklearn.manifold.TSNE(perplexity=30, random_state=2026)` and `from umap import UMAP; UMAP(random_state=2026)`. Label axes, include a legend, and add a title to each panel."

**Why the format request matters.** Without an explicit directive, some AI platforms (LibreChat in particular) wrap long code answers in a Word document — which you cannot paste into Colab.

**Why this is the lab's hardest AI test.** A naive AI answer will apply t-SNE and UMAP directly to all 3,051 genes — slower, noisier, and against the recommended PCA-first workflow. A good answer will reduce to the top ~30 PCs first. *Don't tell the AI which; check what it does.*

In [ ]:
# YOUR CODE HERE (AI-generated or your own)


### AI Critique Checklist

Paste a brief description of what the AI's code produced and complete the table.

**What the AI's code did (one or two sentences):**


| Criterion | Pass / Fail / Partial | Notes |
|---|---|---|
| **Did the AI run PCA first before t-SNE / UMAP?** This is THE most common error and the central test of this exercise. | | |
| Did it transpose the CSV correctly (genes × samples → samples × genes)? | | |
| Did it use `StandardScaler` (or equivalent) before PCA? | | |
| Did it set `random_state=2026` for both t-SNE and UMAP? | | |
| Are both panels labeled with axes, a legend, and a panel title? | | |
| Do both methods show ALL/AML separation? | | |

**One-sentence overall assessment** — did the AI use the right preprocessing without being told?

---

## Method Comparison Summary

| Property | PCA | t-SNE | UMAP |
|----------|-----|-------|------|
| Linear? | Yes | No | No |
| Preserves global structure? | Yes | No | Partially |
| Preserves local structure? | Partially | Yes | Yes |
| Deterministic? | Yes | No | No |
| Interpretable axes? | Yes (loadings) | No | No |
| Needs parameter tuning? | No (beyond n_components) | Yes (perplexity) | Yes (n_neighbors, min_dist) |
| Scales to large n? | Yes | Poorly | Yes |

---

## Wrap-Up Questions

1. All three methods separate ALL from AML on this dataset. Does the existence of that separation imply the underlying structure is linear, nonlinear, or could it be either?

2. The reproducibility cell in Part 6 showed that two t-SNE runs at different seeds produce different layouts. Use the seed-comparison framing to *quantify* the difference: which qualitative features persist across seeds, and which do not?

3. A collaborator shows you a UMAP plot and says "look how far apart these two populations are." What do you say?

4. When would t-SNE or UMAP reveal structure that PCA misses entirely? When would the reverse be true?

---

## Looking Ahead

PCA and UMAP show that ALL and AML are well separated. But dimension reduction is descriptive — it does not give you a predictive model or tell you which specific genes matter most. In **Lab 4**, you will use penalized regression to find a small set of genes that can *predict* leukemia type, and compare that gene list to the FDR list from Lab 2.